# EDA и анализ качества данных

## Цель ноутбука
- Загрузить данные и проверить структуру
- Проверить наличие пропусков и дубликатов
- Проанализировать целевую переменную `Exited`
- Подготовить списки признаков для дальнейшего использования

## Используемые данные
- Датасет: `Churn_Modelling.csv`
- Источник: синтетические данные банка

## Основные выводы
- Данные не содержат критических пропусков
- Целевая переменная несбалансирована
- Требуется очистка от неинформативных колонок

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Папка для артефактов (на уровень выше от notebooks)
ARTIFACTS_DIR = Path.cwd().parent / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Артефакты будут сохранены в: {ARTIFACTS_DIR.absolute()}')

In [ ]:
DATA_PATH = Path.cwd().parent / 'data' / 'raw' / 'Churn_Modelling.csv'
print('DATA_PATH =', DATA_PATH)
print('Файл существует:', DATA_PATH.exists())

## 1. Загрузка данных

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print('Размер датасета:', df.shape)
print('\nТипы данных:')
print(df.dtypes)

In [ ]:
df.info()

## 2. Первичный анализ качества данных

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
print('Пропуски:')
print(missing)

In [ ]:
duplicates = df.duplicated().sum()
print('Количество дубликатов:', duplicates)

In [ ]:
df.describe(include='all').T

## 3. Анализ целевой переменной

In [ ]:
target_counts = df['Exited'].value_counts()
target_ratio = df['Exited'].value_counts(normalize=True)

print('Распределение классов:')
print(target_counts)
print('\nДоли классов:')
print(target_ratio)

In [ ]:
# ВАЖНЫЙ ГРАФИК: Распределение целевой переменной
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(data=df, x='Exited', ax=ax1)
ax1.set_title('Распределение целевой переменной')
ax1.set_xlabel('Exited')
ax1.set_ylabel('Количество')

df['Exited'].value_counts().plot.pie(autopct='%1.1f%%', ax=ax2)
ax2.set_title('Доля классов')
ax2.set_ylabel('')

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / '01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Сохранён: {ARTIFACTS_DIR / "01_target_distribution.png"}')

## 4. Подготовка списка признаков

In [ ]:
drop_columns = ['RowNumber', 'CustomerId', 'Surname']
target = 'Exited'

numeric_features = [
    'CreditScore', 'Age', 'Tenure', 'Balance',
    'NumOfProducts', 'EstimatedSalary'
]

categorical_features = ['Geography', 'Gender']
binary_features = ['HasCrCard', 'IsActiveMember']

all_features = numeric_features + categorical_features + binary_features

print('Числовые признаки:', numeric_features)
print('Категориальные признаки:', categorical_features)
print('Бинарные признаки:', binary_features)

In [ ]:
df_clean = df.drop(columns=drop_columns).drop_duplicates().copy()
print('Размер после очистки:', df_clean.shape)
df_clean.head()

## 5. Выводы по EDA

- Критических пропусков в данных не обнаружено
- Явных дубликатов практически нет
- Целевая переменная `Exited` несбалансирована
- Требуется удалить неинформативные колонки: `RowNumber`, `CustomerId`, `Surname`
- Признаки имеют смешанные типы: числовые, категориальные, бинарные